# PocketTTS: Nigerian English, one language at a time

Fine-tune the [expressive PocketTTS fork](https://github.com/jhony112/pocket-tts-expressive/tree/develop/expressive-streaming) on Nigerian English speech. Run cells from top to bottom in a **GPU Colab runtime**. Checkpoints and prepared data are saved in your Google Drive so you can resume later.

Nigerian English is an English variety, so this notebook uses PocketTTS's **same-language `finetune.yaml`** and keeps its English tokenizer. It does not reset text embeddings. Emotion steering is optional at playback; the public accent datasets below do not provide emotion labels, so this training does not teach new emotions.

**Before training:** listen to the pretrained English sample in the playback cell. This first public-data run is a pilot; the datasets below are far smaller than the 100+ hours [PocketTTS recommends for a basic model](https://github.com/kyutai-labs/pocket-tts/blob/main/training/README.md). Evaluate pronunciation and accent by ear and with held-out text before investing in a long run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install the fork and check your GPU

This notebook tracks `develop/expressive-streaming`. The printed commit identifies exactly what Colab ran. Re-running setup in a fresh runtime will use the newest commit on that branch.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/jhony112/pocket-tts-expressive.git'
BRANCH = 'develop/expressive-streaming'
REPO = Path('/content/pocket-tts-expressive')
ROOT = Path('/content/drive/MyDrive/pocket-tts-expressive/nigerian-english')
ROOT.mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Fork commit:', commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
uv = Path(sys.executable).parent / 'uv'
subprocess.run([str(uv), 'sync', '--python', sys.executable], cwd=REPO, check=True)
PY = REPO / '.venv/bin/python'
subprocess.run([str(uv), 'pip', 'install', '--python', str(PY), 'datasets>=3,<5'], check=True)
subprocess.run([str(PY), '-c', 'import torch; assert torch.cuda.is_available(), "Select a GPU runtime in Colab"; print(torch.cuda.get_device_name(0)); print(round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), "GiB")'], cwd=REPO, check=True)

In [ ]:
from IPython.display import Audio as PlayAudio, display
baseline = ROOT / 'pretrained_english.wav'
if not baseline.exists():
    subprocess.run([str(PY), '-m', 'pocket_tts', 'generate', '--language',
                    'english_2026-01', '--voice', 'alba', '--device', 'cuda',
                    '--text', 'Good afternoon. I understand your concern, and I will help you now.',
                    '--output-path', str(baseline)], cwd=REPO, check=True)
display(PlayAudio(filename=str(baseline)))

## 2. Download Nigerian English recordings

Sources: [Nigerian Common Voice, English split](https://huggingface.co/datasets/benjaminogbonna/nigerian_common_voice_dataset) and [Nigerian accented English](https://huggingface.co/datasets/benjaminogbonna/nigerian_accented_english_dataset). Their revisions are pinned below. Check the dataset cards and licences for your intended use. The script saves 24 kHz WAV files and manifest rows in Drive, filters short/long clips, and removes exact audio duplicates. A tiny dataset is suitable for a smoke test, not a reliable new voice.

In [ ]:
import textwrap
prepare = REPO / 'nigerian_english_prepare.py'
prepare.write_text(textwrap.dedent(r'''
import hashlib, io, json, math, re, unicodedata
from collections import Counter
from pathlib import Path
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from datasets import Audio, load_dataset

ROOT = Path('/content/drive/MyDrive/pocket-tts-expressive/nigerian-english')
AUDIO = ROOT / 'audio'
AUDIO.mkdir(parents=True, exist_ok=True)
SOURCES = [
    ('common_voice_en', 'benjaminogbonna/nigerian_common_voice_dataset', 'english', '97c7f160c0d563339d2f32d55945abc406696cf2'),
    ('accented_en', 'benjaminogbonna/nigerian_accented_english_dataset', None, '23063ce074a2e7b930475d7e776e505c2027d69d'),
]
rows, seen, counts = [], set(), Counter()
for label, repo, config, revision in SOURCES:
    for source_split in ('train', 'validation'):
        ds = load_dataset(repo, name=config, split=source_split, revision=revision, streaming=True)
        ds = ds.cast_column('audio', Audio(decode=False))
        for row in ds:
            text = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(row.get('sentence') or ''))).strip()
            if len(text.split()) < 3:
                counts['short transcript'] += 1
                continue
            audio = row.get('audio') or {}
            raw = audio.get('bytes')
            if raw is None and audio.get('path'):
                raw = Path(audio['path']).read_bytes()
            if not raw:
                counts['missing audio'] += 1
                continue
            digest = hashlib.sha256(raw).hexdigest()
            if digest in seen:
                counts['duplicate'] += 1
                continue
            try:
                wav, sr = sf.read(io.BytesIO(raw), dtype='float32', always_2d=True)
                wav = wav.mean(axis=1)
                duration = len(wav) / sr
                if not 2 <= duration <= 30 or not np.isfinite(wav).all():
                    counts['duration or audio'] += 1
                    continue
                if sr != 24000:
                    g = math.gcd(sr, 24000)
                    wav = resample_poly(wav, 24000 // g, sr // g).astype('float32')
                path = AUDIO / f'{digest}.wav'
                sf.write(path, wav, 24000, subtype='PCM_16')
            except Exception as exc:
                counts['decode failed'] += 1
                print('Skipped undecodable audio:', type(exc).__name__)
                continue
            seen.add(digest)
            speaker = f"{label}:{row.get('client_id') or digest[:16]}"
            rows.append({'path': str(path), 'duration': len(wav)/24000, 'transcript': text,
                         'speaker': speaker, 'source': label, 'source_split': source_split})
            counts[f'{label}/{source_split}'] += 1

# Group by speaker to keep a speaker out of both sets. This is a pilot split.
speakers = sorted({r['speaker'] for r in rows})
assert len(speakers) >= 2, 'Need at least two speakers for train and validation.'
valid_speakers = {s for s in speakers if int(hashlib.sha256(s.encode()).hexdigest(), 16) % 10 == 0}
if not valid_speakers:
    valid_speakers.add(speakers[0])
if len(valid_speakers) == len(speakers):
    valid_speakers.remove(speakers[-1])
for split in ('train', 'valid'):
    selected = [r for r in rows if (r['speaker'] in valid_speakers) == (split == 'valid')]
    assert selected, f'Empty {split} split'
    manifest = ROOT / f'{split}.jsonl'
    with manifest.open('w') as f:
        for row in selected:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(split, len(selected), 'clips', round(sum(r['duration'] for r in selected)/3600, 2), 'hours', manifest)
print('Filter counts:', dict(counts))
'''))
subprocess.run([str(PY), str(prepare)], cwd=REPO, check=True)

## 3. Align words with audio

PocketTTS training needs timestamps for each word. The English Wav2Vec2 aligner works as a starting point for Nigerian English; inspect its failures because accented speech and code switching can reduce alignment quality. Each output is written to a temporary file and renamed only after the command succeeds.

In [ ]:
import subprocess
for split in ('train', 'valid'):
    source = ROOT / f'{split}.jsonl'
    target = ROOT / f'{split}_aligned.jsonl'
    if target.exists():
        print('Already aligned:', target)
        continue
    temporary = ROOT / f'{split}_aligned.partial.jsonl'
    subprocess.run([str(PY), '-m', 'training.scripts.align_data', str(source), str(temporary),
                    '--model', 'facebook/wav2vec2-base-960h', '--device', 'cuda', '--resume'], cwd=REPO, check=True)
    temporary.replace(target)
    print('Aligned:', target)

In [ ]:
import json
for split in ('train', 'valid'):
    path = ROOT / f'{split}_aligned.jsonl'
    rows = [json.loads(line) for line in path.open()]
    good = [r for r in rows if len(r.get('words', [])) >= 3]
    print(split, len(good), '/', len(rows), 'clips with at least 3 aligned words')
    assert len(good) >= (10 if split == 'train' else 1), 'Too few aligned clips; inspect data/alignment first.'

## 4. Fine-tune the English teacher

Checkpoints go to Drive. The default 1,000 steps is a **pilot**, not a claim of production quality. Colab runtimes vary; a smaller GPU may need a smaller batch. Re-run this cell after interruption to resume from the last checkpoint in the same run folder.

In [ ]:
import subprocess, yaml
gpu_gib = float(subprocess.check_output([str(PY), '-c', 'import torch; print(torch.cuda.get_device_properties(0).total_memory/2**30)'], cwd=REPO, text=True))
batch = 64 if gpu_gib >= 70 else 32 if gpu_gib >= 38 else 8 if gpu_gib >= 16 else 4
accumulation = 64 // batch
config = yaml.safe_load((REPO / 'training/configs/finetune.yaml').read_text())
config['data']['train_jsonl'] = str(ROOT / 'train_aligned.jsonl')
config['data']['valid_jsonl'] = str(ROOT / 'valid_aligned.jsonl')
config['run_dir'] = str(ROOT / 'run')
config['batch_size'] = batch
config['grad_accum_steps'] = accumulation
config['max_steps'] = 1000
config['valid_freq'] = 250
config['ckpt_freq'] = 250
config['sample_freq'] = 250
config['sample_sentences'] = [
    'Good afternoon. How can I help you today?',
    'I understand your concern. Let me check your account now.',
    'The transfer has not gone through yet, but your balance is safe.',
]
config['data']['loader_procs'] = 2
cfg_path = ROOT / 'finetune_nigerian_english.yaml'
cfg_path.write_text(yaml.safe_dump(config, sort_keys=False))
print('Batch:', batch, 'accumulation:', accumulation, 'effective batch:', batch*accumulation)
print('Config:', cfg_path)
subprocess.run([str(PY), '-c', 'from training.args import load_args; import sys; print(load_args(sys.argv[1]))', str(cfg_path)], cwd=REPO, check=True)

In [ ]:
# This may run for hours. Colab can disconnect; rerun to resume from Drive.
subprocess.run([str(PY), 'training/train.py', str(cfg_path)], cwd=REPO, check=True)

## 5. Listen and compare

The training run writes samples under `run/samples/`. You can play these while training is in progress. For a fresh sentence, use the latest checkpoint and a clean Nigerian English prompt WAV of at least a few seconds. The prompt is for voice conditioning; use a recording you have permission to use.

In [ ]:
from IPython.display import Audio as PlayAudio, display
samples = sorted((ROOT / 'run/samples').glob('**/*.wav'))
print('Saved training samples:', len(samples))
for sample in samples[-3:]:
    print(sample)
    display(PlayAudio(filename=str(sample)))

In [ ]:
# Optional: after a checkpoint exists, set PROMPT_WAV to one of your clean recordings.
PROMPT_WAV = ''  # e.g. str(ROOT / 'audio' / 'your_prompt.wav')
checkpoints = sorted((ROOT / 'run').glob('checkpoint_*.pt'))
print('Latest checkpoint:', checkpoints[-1] if checkpoints else 'none yet')
if PROMPT_WAV and checkpoints:
    output = ROOT / 'nigerian_english_test.wav'
    subprocess.run([str(PY), '-m', 'pocket_tts', 'generate', '--config',
                    'pocket_tts/config/english_2026-04_24l.yaml', '--checkpoint',
                    str(checkpoints[-1]), '--voice', PROMPT_WAV,
                    '--text', 'Good afternoon. I understand your concern, and I will help you now.',
                    '--output-path', str(output)], cwd=REPO, check=True)
    display(PlayAudio(filename=str(output)))

## What this run proves

A successful run proves that the fork can prepare Nigerian English speech, align it, fine-tune the English teacher, save checkpoints, and synthesize test audio. It does **not** prove accent quality, safe call-center deployment, or emotion control. Compare held-out Nigerian English sentences with the original English model and listen for pronunciation, clarity, speaker consistency, and word omissions. Add more licensed, accurately transcribed Nigerian English speech before a serious training run.

To train another language later, make a **separate notebook** with its own data, tokenizer choice, alignment model, and Drive run directory. Do not combine languages into this pilot by accident.